# Lab 4: Twin families, resemblance, and probability

## Learning objectives

By the end of this lab, you will be able to:

- identify the unit of observation, key, and nesting of participant, participant-visit, and derived pair-level tables;
- transform participant records into a unique twin-pair table and audit how each step changes the denominator;
- distinguish recent use, regular use, and unavailable substance-use data while preserving all three states;
- calculate and interpret marginal, joint, and conditional probabilities and state the denominator used by each;
- apply the complement rule and distinguish mutually exclusive events from independent events;
- assess independence descriptively by comparing an observed joint probability with the value expected under independence;
- interpret paired scatterplots and correlations while distinguishing correlation from agreement; and
- state inferential limitations and family-data-governance concerns raised by linked family records.

### Connection to the Lab 4 warm-up

In the completed **Lab 4 Warm Up: Reading twin-pair records and resemblance**, you read a twin-pair row and the UPPS-P negative-urgency resemblance output. In this lab, you will construct that pair table and reproduce the negative-urgency result. The warm-up prepared you to read the output; this lab shows how the data and code produce it.

### Dataset provenance and connection to Exercise 4

The lab files are synthetic instructional data modeled on real ABCD table and variable structures.

Exercise 4 introduced scatterplots, correlation, and the idea that the suitability of a score depends on its distribution and measurement properties. Lab 4 applies those ideas to genetically related participants and adds probability within twin pairs.

These are not official ABCD observations or national prevalence estimates. Results from these files must never be reported as ABCD findings.

**One caution applies throughout and is not repeated in full at every section.** Every result in this notebook is descriptive. Nothing here identifies a genetic or environmental mechanism, and nothing here establishes that one variable caused another. Section 7 states the limits in full.

### Evidence labels

This lab marks the basis for each analytic choice, as Lab 2 did:

- **DEAP** means a value or category comes from the official data dictionary;
- **DOC** means it comes from study documentation;
- **LITERATURE** means it comes from published research; and
- **RESEARCHER DECISION** means we chose it, and the choice must be stated and defended.

A threshold that separates "regular use" from other use is a **RESEARCHER DECISION**. It is fixed before any analysis and is never adjusted to produce a preferred result.
Documentation: [ABCD DEAP](https://abcd.deapscience.com/), [ABCD genetics and relatedness](https://docs.abcdstudy.org/latest/documentation/non_imaging/gn.html), and [ABCD substance-use measures](https://docs.abcdstudy.org/latest/documentation/non_imaging/su.html). The CSVs in this lab are synthetic instructional data.


### Using Copilot as a tutor

You may use GitHub Copilot while working in this notebook. Use it to support your learning by asking it to:

- explain what a line of code does;
- clarify a concept in beginner-friendly language;
- interpret an error message;
- provide a hint or suggest what to check next; or
- help you evaluate whether your reasoning makes sense.

You are still responsible for running the code, examining the outputs, and understanding each step. Do not ask an AI tool to complete the notebook or fill in all the TODOs for you.

Complete the Canvas reflection independently using your notebook and permitted documentation. Do not use generative AI to answer, draft, rewrite, or check assessment responses, and do not paste assessment questions into an AI tool. Approved accessibility tools, spelling and grammar checkers, and instructor-authorized translation tools are permitted.

When you are unsure about a **Your Turn** cell, first compare it with the preceding **Demo** and identify what needs to change. Then use Copilot for explanation or a hint if needed.

## 1. Research question and student choices

This lab asks two Module 4 questions:

> Within monozygotic (MZ) and dizygotic (DZ) twin pairs, are one twin's recent substance use and the co-twin's recent substance use independent?
>
> How does within-pair resemblance on UPPS-P negative urgency compare between MZ and DZ twins?

You make one analytic choice: choose alcohol or cannabis below for the probability analysis in Sections 4 and 5. Section 6 is a required, completed UPPS-P negative-urgency resemblance demonstration that reproduces the output from the warm-up. Section 8 offers an optional trait transfer task after the required work is complete.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Paths are relative to this notebook in labs/
gn_y_genrel = pd.read_csv("../data/labs_04_05/gn_y_genrel.csv")
su_y_tlfb = pd.read_csv("../data/labs_04_05/su_y_tlfb.csv")
mh_y_upps = pd.read_csv("../data/labs_04_05/mh_y_upps.csv")
nc_y_nihtb = pd.read_csv("../data/labs_04_05/nc_y_nihtb.csv")

source_table_summary = pd.DataFrame({
    "table": ["gn_y_genrel", "su_y_tlfb", "mh_y_upps", "nc_y_nihtb"],
    "rows": [len(gn_y_genrel), len(su_y_tlfb), len(mh_y_upps), len(nc_y_nihtb)],
    "columns": [gn_y_genrel.shape[1], su_y_tlfb.shape[1],
                mh_y_upps.shape[1], nc_y_nihtb.shape[1]],
})

display(source_table_summary)

In [ ]:
# ---- CHOOSE A SUBSTANCE TO ANALYZE --------------------------------------
# Copy one option exactly from the list.
# "su_y_tlfb__mj__3mo_ud"  = cannabis
# "su_y_tlfb__alc__3mo_ud" = alcohol
MY_SUBSTANCE = "su_y_tlfb__mj__3mo_ud"
# ------------------------------------------------------------------------

TWIN_SESSION = "ses-07A"
REGULAR_USE_DAYS = 13             # RESEARCHER DECISION, see Section 4

variable_documentation = {
    "su_y_tlfb__mj__3mo_ud": {"label": "Marijuana: Number of use days (Last 3 months)"},
    "su_y_tlfb__alc__3mo_ud": {"label": "Alcohol: Number of use days (Last 3 months)"},
}

assert MY_SUBSTANCE in variable_documentation, f"Unknown substance: {MY_SUBSTANCE}"
assert MY_SUBSTANCE in su_y_tlfb.columns, f"{MY_SUBSTANCE} is not a column of su_y_tlfb"

doc = variable_documentation[MY_SUBSTANCE]

print("Assigned variable :", MY_SUBSTANCE)
print("Label             :", doc["label"])
print("Source table      : su_y_tlfb")
print("Respondent        : youth")
print("Reference period  : past 90 days")
print("Unit              : days")
print("Missing codes     : a blank value is unavailable, which is not the same as zero")
print("Regular-use rule  : at least", REGULAR_USE_DAYS, "days in 90 (RESEARCHER DECISION)")
print("Status            : synthetic instructional data, not official study values")

## 2. What is a row? Unit of observation, keys, and nesting

This section introduces three related ideas that determine how a table can be analyzed: the unit of observation, the key, and the way records are nested.

### Unit of observation

The **unit of observation** is what one row represents. A participant table has one row per person. A participant-visit table has one row per person at one visit.

| Table | Unit of observation |
| --- | --- |
| `gn_y_genrel` | Participant |
| `su_y_tlfb` | Participant-visit |
| `mh_y_upps` | Participant-visit |
| `nc_y_nihtb` | Participant-visit |

### Keys

A **key** is the column, or combination of columns, that uniquely identifies each row. The key must match the table's unit of observation.

| Table | Key |
| --- | --- |
| `gn_y_genrel` | `participant_id` |
| `su_y_tlfb` | `participant_id` + `session_id` |
| `mh_y_upps` | `participant_id` + `session_id` |
| `nc_y_nihtb` | `participant_id` + `session_id` |

### Demo: use identifier columns to determine what each row represents

In [ ]:
# Demo: run this completed cell.

print("gn_y_genrel :", len(gn_y_genrel), "rows for",
      gn_y_genrel["participant_id"].nunique(), "participants")

print("su_y_tlfb   :", len(su_y_tlfb), "rows for",
      su_y_tlfb["participant_id"].nunique(), "participants across",
      su_y_tlfb["session_id"].nunique(), "sessions")

display(gn_y_genrel.head(3))
display(su_y_tlfb.head(3))

### What to notice

`gn_y_genrel` is keyed by `participant_id` alone, so one row is one person.
The participant-visit tables require both `participant_id` and `session_id`, so one
row is one person at one visit. The same participant can therefore appear in several
rows without those rows being duplicates.

### Your Turn 1: Nesting

**Nesting** describes how smaller units are grouped within larger units. Here,
participants belong to birth sets, and birth sets belong to families. One family can
contain multiple birth sets, and one birth set can contain multiple participants.

#### Where these identifiers came from

Family and birth-set identifiers are created by linking enrollment, biological, and
genetic records. These links may be incomplete, so a missing family or relative
identifier means that no qualifying link was recorded, not that the participant has
no relatives. 

### Your Turn

Count the levels of nesting in the relatedness table: participants, families, and
birth sets. A family may contain more than one birth set.

In [ ]:
# Your Turn 1: count participants, families, and birth sets.
# TODO: replace each None with the correct expression.

n_participants = None    # TODO: count unique participant_id
n_families = None        # TODO: count unique gn_y_genrel_id__fam
n_birth_sets = None      # TODO: count unique gn_y_genrel_id__birth

assert all(isinstance(value, (int, np.integer)) for value in (n_participants, n_families, n_birth_sets)), (
    "Complete the three nesting counts before continuing."
)

nesting_summary = pd.DataFrame({
    "level": ["participants", "families", "birth sets"],
    "count": [n_participants, n_families, n_birth_sets],
})

display(nesting_summary)

### What to notice

There are more birth sets than families, and more participants than birth sets. A
family containing two separate birth events holds participants who are related
but were not born together. Birth timing is what separates twins from other
siblings. 

### Prepare for the Canvas reflection 1: table unit of observation and nesting

Report the number of participants, families, and birth sets. Explain what one row represents in `gn_y_genrel` compared with `su_y_tlfb`, naming the column or columns that told you, and why there are more birth sets than families. These counts do not depend on the choices you made.

## 3. From participant rows to twin-pair rows

In `gn_y_genrel`, a related participant is linked by
`gn_y_genrel_id__paired__01`. The link is **reciprocal**: if A names B, then B
names A. Each pair therefore appears **twice**, once from each member's
perspective. This is the participant-to-pair transformation behind the pair row
and pair-count denominator you read in the warm-up.

The zygosity column records the genetically inferred relationship. **DEAP**

| Level | Meaning |
| --- | --- |
| 1 | Monozygotic |
| 2 | Dizygotic |
| 3 | Siblings |
| 444 | Not available (twins/sibs, genetic_pi_hat not calculated) |

A blank `gn_y_genrel_id__paired__01` means **no qualifying paired relative is
recorded in this field**. That can happen for several reasons, including
unavailable genotype data, quality-control exclusions, or a relative who was
never enrolled. It does not establish that the participant has no relatives.

### Demo: one pair, two rows

In [ ]:
# Demo: run this completed cell.
example = gn_y_genrel.loc[gn_y_genrel["gn_y_genrel_zyg__01"] == 1].iloc[0]
partner_id = example["gn_y_genrel_id__paired__01"]

both_rows = gn_y_genrel.loc[
    gn_y_genrel["participant_id"].isin([example["participant_id"], partner_id]),
    ["participant_id", "gn_y_genrel_id__fam", "gn_y_genrel_id__birth",
     "gn_y_genrel_id__paired__01", "gn_y_genrel_pihat__01", "gn_y_genrel_zyg__01"],
]

display(both_rows)

print("Participants by zygosity level:")
display(gn_y_genrel["gn_y_genrel_zyg__01"].value_counts(dropna=False).sort_index())

### What to notice

Both rows describe one relationship. They carry the same family identifier, the
same birth-set identifier, the same relatedness estimate, and the same zygosity,
and each names the other participant.

The value counts are **participants**, not pairs. Because every pair appears
twice, the number of monozygotic pairs is half the number of monozygotic
participants. Reporting the participant count as a pair count is the most common
error in this analysis.

### Demo: quality control before deduplication

Reciprocity alone is not enough. Two linked rows should also agree on the values
describing their shared relationship, and no participant should end up in more
than one retained pair. These checks are completed for you.

One subtlety decides how the check is written. Twins share a birth event, so a
twin pair must agree on **both** family and birth set. A non-twin sibling pair
shares a family but was born at a different time, so its two rows agree on family
and **disagree** on birth set, by design. Because the pair table keeps only
monozygotic and dizygotic pairs, the check below runs on twin links only. Running
it over sibling links too would report a disagreement for every sibling pair and
hide any real problem.

In [ ]:
# Demo: run this completed cell.
assert gn_y_genrel["participant_id"].is_unique, "participant_id is not a unique key"

# Twin links only, matching the pairs that build_pair_table() retains.
linked = gn_y_genrel.loc[
    gn_y_genrel["gn_y_genrel_id__paired__01"].notna()
    & gn_y_genrel["gn_y_genrel_zyg__01"].isin([1, 2])
].copy()
by_id = gn_y_genrel.set_index("participant_id")

partner = by_id.reindex(linked["gn_y_genrel_id__paired__01"])

qc = pd.DataFrame({
    "check": [
        "link points back to the original participant",
        "partner rows agree on family id",
        "partner rows agree on birth-set id (twins share a birth)",
        "partner rows agree on zygosity",
        "partner rows agree on relatedness estimate",
    ],
    "rows checked": [len(linked)] * 5,
    "disagreements": [
        int((partner["gn_y_genrel_id__paired__01"].to_numpy()
             != linked["participant_id"].to_numpy()).sum()),
        int((partner["gn_y_genrel_id__fam"].to_numpy()
             != linked["gn_y_genrel_id__fam"].to_numpy()).sum()),
        int((partner["gn_y_genrel_id__birth"].to_numpy()
             != linked["gn_y_genrel_id__birth"].to_numpy()).sum()),
        int((partner["gn_y_genrel_zyg__01"].to_numpy()
             != linked["gn_y_genrel_zyg__01"].to_numpy()).sum()),
        # Floating-point values are compared with a tolerance, not for exact equality.
        int((~np.isclose(partner["gn_y_genrel_pihat__01"].to_numpy(),
                         linked["gn_y_genrel_pihat__01"].to_numpy())).sum()),
    ],
})

display(qc)

assert qc["disagreements"].eq(0).all(), (
    "Resolve pair-link disagreements before constructing the pair table."
)

print("All twin links passed. Sibling links are excluded from this check because")
print("siblings share a family but not a birth event.")

### Demo: build the unique pair table

Deduplicating a reciprocal link requires a **canonical ordering**. We sort the two
identifiers so each pair produces the same key from either direction, keep one row
per key, then sort the result before numbering so that the same twins always
receive the same `pair_id` no matter how the source file was ordered.

In [ ]:
# Demo: run this completed cell. Source DataFrames are not modified.
def build_pair_table(relatedness, keep_zygosity=(1, 2)):
    """Return one row per genetically related pair, with a reproducible pair_id."""
    rows = relatedness.loc[
        relatedness["gn_y_genrel_id__paired__01"].notna()
        & relatedness["gn_y_genrel_zyg__01"].isin(keep_zygosity)
    ].copy()

    first = rows[["participant_id", "gn_y_genrel_id__paired__01"]].min(axis=1)
    second = rows[["participant_id", "gn_y_genrel_id__paired__01"]].max(axis=1)

    pairs = pd.DataFrame({
        "twin_a": first,
        "twin_b": second,
        "zygosity": rows["gn_y_genrel_zyg__01"].to_numpy(),
        "family_id": rows["gn_y_genrel_id__fam"].to_numpy(),
    }).drop_duplicates(subset=["twin_a", "twin_b"])

    # Sort before numbering so pair_id does not depend on source row order.
    pairs = pairs.sort_values(["twin_a", "twin_b"]).reset_index(drop=True)
    pairs["pair_id"] = ["pair-" + str(i + 1).zfill(5) for i in range(len(pairs))]
    pairs["zygosity_label"] = pairs["zygosity"].map({1: "MZ", 2: "DZ"})
    return pairs[["pair_id", "family_id", "twin_a", "twin_b", "zygosity", "zygosity_label"]]


pair_table = build_pair_table(gn_y_genrel)

# No participant may appear in more than one retained pair.
members = pd.concat([pair_table["twin_a"], pair_table["twin_b"]])
assert members.is_unique, "a participant appears in more than one pair"

display(pair_table.head(3))
display(pair_table["zygosity_label"].value_counts())

### Demo: denominator audit

In [ ]:
# Demo: run this completed cell.
audit = pd.DataFrame({
    "step": [
        "participants in gn_y_genrel",
        "participants with a recorded paired relative",
        "participants in MZ or DZ pairs",
        "unique MZ or DZ pairs",
        "MZ pairs",
        "DZ pairs",
    ],
    "n": [
        len(gn_y_genrel),
        int(gn_y_genrel["gn_y_genrel_id__paired__01"].notna().sum()),
        len(members),
        len(pair_table),
        int((pair_table["zygosity_label"] == "MZ").sum()),
        int((pair_table["zygosity_label"] == "DZ").sum()),
    ],
})

display(audit)

### What to notice

Filtering to monozygotic and dizygotic pairs removes sibling pairs and every
participant without a recorded paired relative. The analytic sample is far smaller
than the cohort, and each step should be reportable.

The number of participants in twin pairs is exactly twice the number of pairs.

### Prepare for the Canvas reflection 2: the pair table and its denominator

Report the value at each step of the denominator audit, then explain in one or two sentences what the zygosity filter removed and why the number of monozygotic participants is not the number of monozygotic pairs.

Assessed in Canvas; no additional notebook response is required.

## 4. Recent use, regular use, and unavailable data

Two different measures are used in this lab.

**Recent use** is any use in the past 90 days: `use days > 0`.

**Regular use** is at least 13 use days in the 90-day window, approximately once
per week on average. This is a **RESEARCHER DECISION**. It is a proxy, not proof
that use occurred in every calendar week, and it is fixed before analysis.

A third state matters just as much. If a participant's use-day value is
**unavailable**, that is not a report of no use. Converting an unavailable value
to zero manufactures nonuse, which is exactly the error Lab 3 warned against.
The supplied data contain some unavailable use-day values at the selected wave, including among twins. Every derivation in this lab produces **three** states.

### Demo: choose a feasible measure for the twin-pair analysis

In [ ]:
# Demo: run this completed cell.
tlfb_twin_wave = su_y_tlfb.loc[su_y_tlfb["session_id"] == TWIN_SESSION].set_index("participant_id")
use_days_twin_wave = tlfb_twin_wave[MY_SUBSTANCE]

# A three-state indicator retains missing values as unavailable.
regular_flag = (use_days_twin_wave >= REGULAR_USE_DAYS).astype("Int64").where(
    use_days_twin_wave.notna()
)

def make_three_state_flag(values, rule):
    flag = pd.Series(pd.NA, index=values.index, dtype="Int64")
    available = values.notna()
    flag.loc[available] = rule(values.loc[available]).astype("Int64")
    return flag

def pair_probability_counts(indicator, zygosity):
    subset = pair_table.loc[pair_table["zygosity_label"] == zygosity]
    a = indicator.reindex(subset["twin_a"]).to_numpy()
    b = indicator.reindex(subset["twin_b"]).to_numpy()
    complete = ~(pd.isna(a) | pd.isna(b))
    a = a[complete]
    b = b[complete]
    directed_denominator = int((a == 1).sum() + (b == 1).sum())
    directed_numerator = int(2 * ((a == 1) & (b == 1)).sum())
    return int(complete.sum()), directed_denominator, directed_numerator

rows = []
for zygosity in ["MZ", "DZ"]:
    for measure, indicator in [
        ("recent use", make_three_state_flag(use_days_twin_wave, lambda values: values > 0)),
        (f"regular use (>= {REGULAR_USE_DAYS} days)", regular_flag),
    ]:
        complete_n, conditional_denominator, conditional_numerator = pair_probability_counts(
            indicator, zygosity
        )
        rows.append({
            "zygosity": zygosity,
            "measure": measure,
            "complete pairs": complete_n,
            "directed records where index twin used (denominator)": conditional_denominator,
            "directed records where index and co-twin used (numerator)": conditional_numerator,
        })

comparison = pd.DataFrame(rows)

print("Prevalence in the full cohort at this session:")
print("  recent use  :", round(float((use_days_twin_wave.dropna() > 0).mean()), 3))
print("  regular use :", round(float(regular_flag.mean()), 3))
print()
display(comparison)

### What to notice

Regular use is substantively meaningful and less common than recent use. The table shows fewer directed records in which the index twin used and fewer in which both twins used. Its MZ-DZ conditional estimates are therefore less stable than the recent-use estimates in this sample. A regular-use comparison remains possible, but calls for more caution.

The threshold stays fixed because its definition was chosen before analysis. This lab uses **recent use** for the main twin-pair probability analysis to give the comparison a larger event count. Recent use and regular use remain different measures and must not be interpreted as interchangeable.

Both indicators preserve unavailable values as missing. Treating unavailable data as nonuse would make either measure misleading.

### Your Turn 2

Derive recent use as a **three-state** variable. The object you create here,
`recent_use`, is the one Section 5 analyses, so its correctness matters.

In [ ]:
# Your Turn 2: derive a three-state recent-use variable.
# States: 1 = used in the past 90 days, 0 = did not use, missing = unavailable.
# TODO: replace each None with the correct expression.

# TODO: build the indicator WITHOUT converting unavailable values to 0.
# Hint: compare to 0 first, then restore missingness with .where(use_days_twin_wave.notna())
recent_use = None

assert isinstance(recent_use, pd.Series), (
    "Complete recent_use before continuing: it must be a pandas Series."
)
assert recent_use.dropna().isin([0, 1]).all(), (
    "recent_use may contain only 0, 1, or missing values."
)
assert recent_use.isna().equals(use_days_twin_wave.isna()), (
    "recent_use must preserve every unavailable use-days value as missing."
)

availability = pd.DataFrame({
    "state": ["used", "did not use", "unavailable"],
    "n": [
        None,   # TODO: count of recent_use equal to 1
        None,   # TODO: count of recent_use equal to 0
        None,   # TODO: count of unavailable values
    ],
})

assert availability["n"].notna().all(), (
    "Complete all three availability counts before continuing."
)

display(availability)

### Prepare for the Canvas reflection 3: measuring use responsibly

Name the substance you chose and report your counts of used, did not use, and unavailable. Then explain the difference between recent use, regular use, and unavailable data, and what would go wrong if `use_days > 0` were written without preserving missingness. Finish by saying what your unavailable count tells you about how confident you can be in the prevalence displayed in the feasibility demo above.

Assessed in Canvas; no additional notebook response is required.

## 5. Probability within twin pairs

All five quantities below begin with the **complete-pair sample**: pairs in which both members have an available value. The marginal and joint probabilities use complete pairs as their denominator. The conditional probability uses a restricted set of directed records from those same complete pairs.

- **Twin A marginal**: among complete pairs, the proportion of twin A members who used. Denominator: complete pairs.
- **Twin B marginal**: the same for twin B members. Denominator: complete pairs.
- **Joint probability**: the proportion of complete pairs in which **both** used. Denominator: complete pairs.
- **Conditional probability**: among directed records in which the index twin used, the proportion whose co-twin also used. Denominator: those directed records only.
- **Joint probability expected under independence**: the product of the two marginal probabilities, $P(A) \times P(B)$.

A twin pair has no natural first member, so the conditional probability is computed on **directed records**: each pair contributes two rows, once with each twin as the index. That makes the quantity symmetric and reproducible.

The numerator and denominator must describe the same unit. Marginal and joint probabilities count complete pairs, while the conditional probability counts eligible directed records. A numerically correct division using the wrong table would answer a different question.

"Twin A" and "twin B" are arbitrary labels produced by sorting identifiers. They carry no substantive meaning.

We are not running a statistical test. We compare an observed joint probability with the value expected under independence and describe the difference.

### Demo: count events in four fictional twin pairs

This small example uses invented pairs so you can see exactly which rows enter each probability. A value of `1` means the twin used the substance; `0` means they did not. Each row is **one complete pair**.

Run the next cell and inspect the displayed table before reading the calculated probabilities. With a partner, count:

1. How many pairs are there?
2. In how many pairs did Twin A use?
3. In how many pairs did both twins use?

For this demonstration, each probability below uses **four pairs** as its denominator. The next demonstration will change the unit and denominator when it asks about a co-twin *given that the index twin used*.

In [ ]:
# Demo: run this completed cell. These four pairs are invented for illustration.

# Demo Part 1: probabilities from pairs

example_pairs = pd.DataFrame({
    "pair_id": ["ex-1", "ex-2", "ex-3", "ex-4"],
    "twin_a_uses": [1, 1, 0, 0],
    "twin_b_uses": [1, 0, 1, 0],
})

display(example_pairs)

p_a = example_pairs["twin_a_uses"].mean()
p_b = example_pairs["twin_b_uses"].mean()
p_joint = (
    (example_pairs["twin_a_uses"] == 1)
    & (example_pairs["twin_b_uses"] == 1)
).mean()

print("P(twin A uses)              =", round(p_a, 3), "   denominator: 4 pairs")
print("P(twin B uses)              =", round(p_b, 3), "   denominator: 4 pairs")
print("P(both use), observed       =", round(p_joint, 3), "  denominator: 4 pairs")
print("P(A) x P(B), if independent =", round(p_a * p_b, 3))

### Demo: the conditional probability in the same fictional example

In [ ]:
# Demo Part 2: conditional probability from directed records

example_directed = pd.concat([
    example_pairs.rename(columns={"twin_a_uses": "index_uses",
                                  "twin_b_uses": "cotwin_uses"}),
    example_pairs.rename(columns={"twin_b_uses": "index_uses",
                                  "twin_a_uses": "cotwin_uses"}),
], ignore_index=True)[["pair_id", "index_uses", "cotwin_uses"]]

display(example_directed)

index_used = example_directed.loc[example_directed["index_uses"] == 1]
p_conditional_example = index_used["cotwin_uses"].mean()

print("Directed records total                    :", len(example_directed), "(2 per pair)")
print("Directed records where the index twin used:", len(index_used), "<- the denominator")
print("Of those, co-twin also used               :", int((index_used['cotwin_uses'] == 1).sum()))
print()
print("P(co-twin uses | index twin uses) =", round(p_conditional_example, 3))

### Figure: the same four pairs, counted two ways

Run the completed cell below. Pair IDs stay in the same order in both panels. A **filled** circle means used (`1`); an **empty** circle means did not use (`0`). In the right panel, each arrow points from the index twin to the co-twin. Faded rows have an index twin who did not use and do not enter the conditional denominator.


In [ ]:
# Demo: visualize how four pairs become eight directed records.
from matplotlib.patches import Circle

fig, (ax_pairs, ax_directed) = plt.subplots(1, 2, figsize=(14, 7.5))

def draw_twin(ax, x, y, role, used, faded=False):
    ax.add_patch(Circle(
        (x, y), 0.20, facecolor="#17657a" if used else "white",
        edgecolor="#263b44", linewidth=1.5, alpha=0.35 if faded else 1,
    ))
    ax.text(x, y, role, ha="center", va="center", fontsize=11,
            fontweight="bold", color="white" if used else "#263b44",
            alpha=0.35 if faded else 1)

# Left: one row per pair. The unit and denominator are pairs.
for row_number, pair in example_pairs.reset_index(drop=True).iterrows():
    y = 3.5 - row_number
    ax_pairs.text(0.15, y, pair["pair_id"], va="center", fontweight="bold")
    draw_twin(ax_pairs, 1.0, y, "A", pair["twin_a_uses"] == 1)
    draw_twin(ax_pairs, 2.0, y, "B", pair["twin_b_uses"] == 1)
    ax_pairs.plot([1.22, 1.78], [y, y], color="#677981", linewidth=1.5)
ax_pairs.set_title("Count pairs", fontsize=14, fontweight="bold", loc="left")
ax_pairs.text(0.15, -0.15, "4 pairs: A uses 2/4; B uses 2/4; both use 1/4",
              fontsize=10, fontweight="bold")
ax_pairs.set_xlim(0, 3.1)
ax_pairs.set_ylim(-0.5, 4.15)

# Right: each pair supplies two rows, one for each index twin.
for pair_number, pair in example_pairs.reset_index(drop=True).iterrows():
    for direction, (index_role, cotwin_role) in enumerate((("A", "B"), ("B", "A"))):
        y = 7.5 - (2 * pair_number + direction)
        index_used = pair[f"twin_{index_role.lower()}_uses"] == 1
        cotwin_used = pair[f"twin_{cotwin_role.lower()}_uses"] == 1
        faded = not index_used
        alpha = 0.35 if faded else 1
        ax_directed.text(0.1, y, pair["pair_id"], va="center",
                         fontweight="bold", alpha=alpha)
        draw_twin(ax_directed, 1.25, y, index_role, index_used, faded)
        draw_twin(ax_directed, 2.6, y, cotwin_role, cotwin_used, faded)
        ax_directed.annotate("", xy=(2.36, y), xytext=(1.49, y),
                             arrowprops={"arrowstyle": "->", "lw": 1.5,
                                         "color": "#263b44", "alpha": alpha})
        ax_directed.text(3.05, y, "excluded" if faded else "eligible",
                         va="center", fontsize=9, alpha=alpha)
ax_directed.set_title("Count eligible directed records", fontsize=14,
                      fontweight="bold", loc="left")
ax_directed.text(0.1, -0.35, "4 eligible of 8 records; co-twin used in 2/4",
                 fontsize=10, fontweight="bold")
ax_directed.set_xlim(0, 4.25)
ax_directed.set_ylim(-0.7, 8.15)

for ax in (ax_pairs, ax_directed):
    ax.set_aspect("equal")
    ax.axis("off")
fig.text(0.5, 0.025, "Filled circle = used (1)     Empty circle = did not use (0)",
         ha="center", fontsize=10)
plt.tight_layout(rect=(0, 0.05, 1, 1), w_pad=2)
plt.show()


The left panel counts **four pairs** for marginal and joint probabilities. The right panel counts **four eligible directed records** for the conditional probability. The both-use pair (`ex-1`) supplies two eligible records, and both have a co-twin who used. The denominators happen to have the same number, but they count different units. For the complement demo below, the left panel also shows one pair with neither using (`ex-4`) and three with at least one using.


### Demo: complements, and the probability of at least one

The complement of an event contains every outcome in which that event does not occur. For a twin pair, "at least one uses" is the complement of "neither uses," so

$$P(\text{at least one uses}) = 1 - P(\text{neither uses}).$$

This rule avoids separately counting pairs in which only A uses, only B uses, or both use.

Do not confuse **mutually exclusive** with **independent**. "Both use" and "neither uses" are mutually exclusive because one pair cannot satisfy both descriptions. Independence asks a different question: whether learning one twin's outcome changes the probability of the other twin's outcome. Independent events may occur together; mutually exclusive events with positive probability cannot be independent.

In [ ]:
# Demo: run this completed cell.
p_neither = ((example_pairs["twin_a_uses"] == 0) & (example_pairs["twin_b_uses"] == 0)).mean()

print("P(neither uses)      =", round(p_neither, 3))
print("P(at least one uses) =", round(1 - p_neither, 3), " <- the complement rule")
print()
print("'Both use' and 'neither uses' are mutually exclusive: they cannot both happen")
print("for the same pair. Mutually exclusive is not the same as independent.")
print("Independence is about whether knowing one outcome changes the probability")
print("of the other.")

### Demo: build the complete-pair and directed tables from your derived variable

In [ ]:
# Demo: run this completed cell. It uses the recent_use variable you created.
def build_directed_pairs(pairs, indicator):
    """Two rows per pair, once with each twin as the index twin."""
    forward = pd.DataFrame({
        "pair_id": pairs["pair_id"].to_numpy(),
        "index_role": "A",
        "index_id": pairs["twin_a"].to_numpy(),
        "cotwin_id": pairs["twin_b"].to_numpy(),
        "zygosity_label": pairs["zygosity_label"].to_numpy(),
        "index_uses": indicator.reindex(pairs["twin_a"]).to_numpy(),
        "cotwin_uses": indicator.reindex(pairs["twin_b"]).to_numpy(),
    })
    backward = pd.DataFrame({
        "pair_id": pairs["pair_id"].to_numpy(),
        "index_role": "B",
        "index_id": pairs["twin_b"].to_numpy(),
        "cotwin_id": pairs["twin_a"].to_numpy(),
        "zygosity_label": pairs["zygosity_label"].to_numpy(),
        "index_uses": indicator.reindex(pairs["twin_b"]).to_numpy(),
        "cotwin_uses": indicator.reindex(pairs["twin_a"]).to_numpy(),
    })
    return pd.concat([forward, backward], ignore_index=True).dropna(
        subset=["index_uses", "cotwin_uses"])


complete_pairs = pair_table.assign(
    twin_a_uses=recent_use.reindex(pair_table["twin_a"]).to_numpy(),
    twin_b_uses=recent_use.reindex(pair_table["twin_b"]).to_numpy(),
).dropna(subset=["twin_a_uses", "twin_b_uses"])

directed_pairs = build_directed_pairs(complete_pairs, recent_use)

# The directed table's key is pair_id plus index_role. Verify it.
assert not directed_pairs.duplicated(["pair_id", "index_role"]).any(), (
    "directed table has duplicate pair_id/index_role keys"
)

print("Pairs in the pair table           :", len(pair_table))
print("Pairs with two available values   :", len(complete_pairs))
print("Directed records                  :", len(directed_pairs))
display(directed_pairs.head(4))

### Your Turn 3

For MZ and DZ pairs separately, calculate both marginal probabilities, the observed joint probability, the symmetric conditional probability, and the joint probability expected under independence. Use the complete-pair or directed-record denominator specified for each quantity.

In [ ]:
# Your Turn 3: probability within twin pairs.
# TODO: replace each None with the correct expression.

results = []
for zygosity in ["MZ", "DZ"]:
    pairs_z = complete_pairs.loc[complete_pairs["zygosity_label"] == zygosity]
    directed_z = directed_pairs.loc[directed_pairs["zygosity_label"] == zygosity]

    # TODO: use pairs_z["twin_a_uses"] and pairs_z["twin_b_uses"]
    # to mirror the fictional-pair demo.
    p_twin_a = None
    p_twin_b = None

    # TODO: proportion of complete pairs in which BOTH twins used
    p_both = None

    # The conditional denominator is the directed records where the index twin used.
    index_used = directed_z.loc[directed_z["index_uses"] == 1]

    # TODO: among those records, calculate the proportion whose co-twin also used
    p_conditional = None

    # TODO: the joint probability expected if the twins were independent
    p_expected_if_independent = None

    results.append({
        "zygosity": zygosity,
        "complete pairs": len(pairs_z),
        "conditional denominator (directed records)": len(index_used),
        "P(A uses)": p_twin_a,
        "P(B uses)": p_twin_b,
        "P(both use)": p_both,
        "P(A) x P(B)": p_expected_if_independent,
        "P(co-twin uses | index uses)": p_conditional,
    })

probability_summary = pd.DataFrame(results)
assert probability_summary[[
    "P(A uses)", "P(B uses)", "P(both use)", "P(A) x P(B)",
    "P(co-twin uses | index uses)",
]].notna().all().all(), (
    "Complete all five probability expressions before continuing to the figure."
)

display(probability_summary)

### Figure: observed joint use and the independence expectation

Run this completed cell after your `probability_summary` is populated. It uses the probabilities you calculated for the substance selected above.


In [ ]:
# Demo: compare observed joint probability with P(A) x P(B) for each group.
plot_summary = probability_summary.set_index("zygosity").loc[["MZ", "DZ"]]
groups = plot_summary.index.to_list()
observed = plot_summary["P(both use)"].astype(float).to_numpy() * 100
expected = plot_summary["P(A) x P(B)"].astype(float).to_numpy() * 100
x = np.arange(len(groups))
width = 0.36

fig, ax = plt.subplots(figsize=(8, 5))
observed_bars = ax.bar(x - width / 2, observed, width,
                       label="Observed P(both use)", color="#17657a",
                       edgecolor="#263b44")
expected_bars = ax.bar(x + width / 2, expected, width,
                       label="Expected under independence",
                       color="white", edgecolor="#263b44", hatch="///")

for bars in (observed_bars, expected_bars):
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.3,
                f"{height:.1f}%", ha="center", va="bottom", fontsize=10)

ax.set_xticks(x, groups)
ax.set_ylabel("Joint probability (%)")
ax.set_ylim(0, max(np.r_[observed, expected]) * 1.3)
ax.set_title("Observed joint use vs. independence expectation")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


**Caption.** The bars compare the observed proportion of complete pairs in which both twins used with $P(A) \times P(B)$, the joint probability expected under independence. This is a descriptive comparison in this sample, **not a statistical test**; it does not account for sampling variability or establish dependence in the wider population.


### What to notice

Compare `P(both use)` with `P(A) x P(B)`. If the observed joint probability is
clearly larger than the independence expectation, this sample shows more joint use
than independence would predict, which suggests positive dependence within pairs.

That is a statement about this sample. A descriptive comparison like this one does
not establish dependence in the wider population, because it does not account for
sampling variability.

Compare the two marginal probabilities within each row as a quality check. Because
twin A and twin B are arbitrary analytic labels created by sorting identifiers,
their marginal probabilities should usually be similar. A large difference would
be a reason to inspect the ordering, the missingness, and the pair-construction
procedure. Similar marginals do not by themselves prove that twins are
interchangeable.

### Prepare for the Canvas reflection 4: marginal and conditional probability

For one zygosity group, report and interpret one marginal probability and the conditional probability. Name the substance and state each denominator. Explain why the denominators represent different units and eligibility conditions, then say whether the conditional denominator is numerically smaller in your results.

Assessed in Canvas; no additional notebook response is required.

### Prepare for the Canvas reflection 5: joint probability and independence

For one zygosity group, report the observed joint probability and the value expected under independence. Explain whether the observations appear independent in this sample and why this comparison is descriptive rather than a population-level statistical test.

Also be ready for two concise probability items:

- use the complement of `P(neither uses)` to identify `P(at least one uses)`; and
- distinguish mutually exclusive events, which cannot occur together, from independent events, where knowing one outcome does not change the probability of the other.

Assessed in Canvas; no additional notebook response is required.

## 6. Paired resemblance

This required demonstration reproduces the UPPS-P negative-urgency twin-pair output from the warm-up. It plots the negative-urgency score for both members of each constructed pair, with twin A on one axis, twin B on the other, and a 1:1 reference line.

### Required demonstration: UPPS-P negative urgency

Leave the default `mh_y_upps__nurg_sum` selection unchanged and run the completed cells below. The normal Run All path produces the MZ and DZ scatterplots, their displayed correlations and pair counts, and the score's spread and number of distinct values.

Return to Figure 2 in the warm-up as you interpret this output. The warm-up prepared you to read its pair rows, points, counts, and correlations; this section shows how the constructed pair table and selected score produce them.

Negative urgency is a UPPS-P impulsivity facet: acting rashly when experiencing strong negative emotion. **DEAP** Higher values indicate more of the reported characteristic; they are not a diagnosis.

The configuration cell retains additional traits only for the optional Section 8 transfer task. No trait change is needed for the required lab or Canvas reflection.

In [ ]:
# ---- REQUIRED DEMONSTRATION: UPPS-P NEGATIVE URGENCY --------------------
# Leave this default unchanged for the required lab and Canvas reflection.
MY_TRAIT = "mh_y_upps__nurg_sum"
# ------------------------------------------------------------------------

# OPTIONAL Section 8 transfer choices; none is required for the lab or Canvas.
# "mh_y_upps__sens_sum"                 = sensation seeking (recommended transfer)
# "nc_y_nihtb__comp__tot__agecor_score" = cognition total composite (optional)
# "nc_y_nihtb__picsq__agecor_score"     = picture sequence memory (optional)
# "nc_y_nihtb__flnkr__agecor_score"     = flanker inhibitory control (optional)

TRAIT_MENU = {
    "mh_y_upps__nurg_sum": ("mh_y_upps", "UPPS-P negative urgency",
        "acting rashly when experiencing strong negative emotion"),
    "mh_y_upps__sens_sum": ("mh_y_upps", "UPPS-P sensation seeking",
        "seeking out novel, intense or exciting experiences"),
    "nc_y_nihtb__comp__tot__agecor_score": ("nc_y_nihtb", "Cognition total composite",
        "overall cognitive performance, scored against same-age peers"),
    "nc_y_nihtb__picsq__agecor_score": ("nc_y_nihtb", "Picture sequence memory",
        "episodic memory for a sequence, scored against same-age peers"),
    "nc_y_nihtb__flnkr__agecor_score": ("nc_y_nihtb", "Flanker inhibitory control",
        "attention and inhibitory control, scored against same-age peers"),
}

assert MY_TRAIT in TRAIT_MENU, f"Choose a trait from TRAIT_MENU, not {MY_TRAIT}"
trait_table, trait_name, trait_meaning = TRAIT_MENU[MY_TRAIT]

print("Resemblance trait :", MY_TRAIT)
print("  name            :", trait_name)
print("  source table    :", trait_table)
print("  measures        :", trait_meaning)

In [ ]:
# Required demonstration: run this completed cell with UPPS-P negative urgency.
trait_source = {"mh_y_upps": mh_y_upps, "nc_y_nihtb": nc_y_nihtb}[trait_table]
trait_wave = trait_source.loc[trait_source["session_id"] == TWIN_SESSION]
assert trait_wave["participant_id"].is_unique, "trait table is not one row per participant here"
trait = trait_wave.set_index("participant_id")[MY_TRAIT].astype(float)

print(f"{trait_name}: n = {trait.notna().sum()}, "
      f"{trait.nunique()} distinct values, "
      f"range {trait.min():.0f} to {trait.max():.0f}, sd = {trait.std():.2f}")

paired_trait = pair_table.assign(
    twin_a_score=trait.reindex(pair_table["twin_a"]).to_numpy(),
    twin_b_score=trait.reindex(pair_table["twin_b"]).to_numpy(),
).dropna(subset=["twin_a_score", "twin_b_score"])

# Use the observed range for the 1:1 line rather than hard-coded limits.
observed_low = min(paired_trait["twin_a_score"].min(), paired_trait["twin_b_score"].min())
observed_high = max(paired_trait["twin_a_score"].max(), paired_trait["twin_b_score"].max())
margin = 0.5

rng = np.random.default_rng(4)
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharex=True, sharey=True)

for ax, zygosity in zip(axes, ["MZ", "DZ"]):
    subset = paired_trait.loc[paired_trait["zygosity_label"] == zygosity]
    # Jitter is applied only to coarse scores, where points would otherwise overlap.
    # Use fixed 0.25 score-unit jitter for traits with fewer than 30 distinct values.
    spacing = 0.25 if trait.nunique() < 30 else 0.0
    jitter_a = subset["twin_a_score"] + rng.uniform(-spacing, spacing, len(subset))
    jitter_b = subset["twin_b_score"] + rng.uniform(-spacing, spacing, len(subset))
    ax.scatter(jitter_a, jitter_b, s=10, alpha=0.35)
    ax.plot([observed_low, observed_high], [observed_low, observed_high], linewidth=1)
    ax.set_xlim(observed_low - margin, observed_high + margin)
    ax.set_ylim(observed_low - margin, observed_high + margin)
    ax.set_aspect("equal", adjustable="box")
    correlation = subset["twin_a_score"].corr(subset["twin_b_score"])
    ax.set_title(f"{zygosity} pairs (n = {len(subset)}), r = {correlation:.3f}")
    ax.set_xlabel(f"Twin A {trait_name}")

axes[0].set_ylabel(f"Twin B {trait_name}")
plt.tight_layout()
plt.show()

### What to notice

Return to Figure 2 in the warm-up. Compare the MZ and DZ panels using both the point clouds and the displayed $r$ values.

**Correlation describes linear resemblance; proximity to the 1:1 line describes agreement.** Use both when reading the panels.

The displayed spread and number of distinct values help you judge whether the score can show resemblance. For coarse scores, jitter makes overlapping points visible for display only; it does not change the scores or correlations. Very restricted variation can limit what the plot can show.

Stronger MZ resemblance in this sample alone does not prove a genetic cause. Twin pairs also share age, household, and other environmental experiences.

### Prepare for the Canvas reflection 6: paired resemblance

Identify UPPS-P negative urgency and what it measures. Report the displayed MZ and DZ pair counts and correlations, then interpret the comparison using the point clouds and $r$ values. Distinguish correlation from agreement, and state one limitation of this comparison.

Assessed in Canvas; no additional notebook response is required.

## 7. Interpretation and family-data governance

Linked family records raise questions that participant-level files do not.

**What these results show.** The probability and paired-resemblance analyses describe patterns in a synthetic instructional dataset built to resemble ABCD table structures.

**What they do not show.** They do not identify genetic or environmental mechanisms, establish causation, or estimate heritability. Greater resemblance among MZ than DZ pairs could reflect many shared differences and does not by itself identify a genetic cause.

**Relational privacy.** Linked family records can permit inferences about a co-twin or other relatives beyond what those individuals directly disclosed, including relatives who are not study participants. Identifying one relative may therefore reveal information about another.

**Absent linkage.** A blank paired-relative link means no qualifying relative is recorded in that field. It is not evidence that the participant has no relatives.

**Reporting.** These are synthetic instructional values. They may not be presented as ABCD findings, cited as evidence about real participants, or used to characterize any real group.

### Prepare for the Canvas reflection 7: limitations and governance

State one inferential limitation of either the probability analysis for your selected substance or the required UPPS-P negative-urgency resemblance demonstration. Then identify one relational-privacy concern raised specifically by linked family records and explain why identifying one relative may reveal information about another.

Also be prepared to explain why an absent paired-relative link is not evidence that a participant has no relatives and why these synthetic results cannot be reported as ABCD findings.

Assessed in Canvas; no additional notebook response is required.

## 8. Optional transfer task

This task is ungraded and follows the required work. You may change the selected substance for the probability analysis and rerun Sections 4 and 5.

For an optional resemblance transfer, change `MY_TRAIT` to UPPS-P sensation seeking (`mh_y_upps__sens_sum`), then rerun the Section 6 configuration and completed demonstration. Compare its score spread, MZ and DZ pair plots, and correlations with the required negative-urgency output.

The optional transfer is not needed for the lab or Canvas reflection. Restore UPPS-P negative urgency before your final Run All.

## 9. Finish the lab and complete the Canvas reflection

Before completing the Canvas reflection:

1. Complete all three required **Your Turn** coding activities.
2. Confirm `MY_SUBSTANCE` matches the substance named in your probability answers, and leave `MY_TRAIT` set to the required UPPS-P negative-urgency demonstration.
3. Run all cells from top to bottom so the outputs are current.
4. Confirm the source DataFrames `gn_y_genrel`, `su_y_tlfb`, `mh_y_upps`, and `nc_y_nihtb` remain unchanged and derived variables are stored in new objects.
5. Confirm unavailable use-day values were preserved rather than converted to zero.
6. Verify that marginal and joint probabilities use complete pairs and that conditional probability uses directed records where the index twin used.
7. Review the seven Canvas reflection-preparation blocks and locate the current output needed for each, including the required negative-urgency resemblance output.
8. Save the notebook for your own continued work; the notebook itself is not collected.

The Canvas reflection covers table units and nesting, pair construction and denominators, three-state use measures, marginal, joint, conditional, complement, and independence concepts, the required negative-urgency paired-resemblance demonstration, and limitations and governance.